# Automated Skill Discovery with RLMs

This notebook accompanies Chapter 11, "Optimizing Coding Agents with DSPy", of *Context Engineering with DSPy*. It runs the example from the section "Automated Skill Discovery with RLMs" (pages 299–301), the first example under "Self-Evolving Agents": a `dspy.RLM` reads past agent sessions, groups them by what the user was asking for, and drafts a `SKILL.md` for the most common request.

## Before you run

- **API key:** `OPENAI_API_KEY`, with access to `openai/gpt-5.6-luna`. Add it to a `.env` file at the repo root (see `.env.example`).
- **Deno:** `dspy.RLM` runs the code it writes in a Deno and Pyodide sandbox. Install Deno in a terminal (not as a notebook cell), as described in the Preface under "Additional Software for Later Chapters" (page xv), then confirm it with `deno --version`. On macOS or Linux:

  ```bash
  curl -fsSL https://deno.land/install.sh | sh
  ```

  On Windows PowerShell, use `irm https://deno.land/install.ps1 | iex`.
- **Cost:** one RLM run over five short transcripts, capped at 8 iterations and 10 sub-LM calls.
- **Install cell:** if you set up the environment with `uv sync --frozen` as described in the Preface, everything is already installed and the install cell below has nothing left to do. With pip or Colab, run the notebook from the `chapter11/` folder so that `../requirements.txt` resolves.

In [ ]:
%pip install -r ../requirements.txt -q

## Discovering new skills from past sessions

When users repeatedly ask an agent for the same kind of help, that's a missing skill. The catch is you don't notice because each session looks different on the surface. A Recursive Language Model (introduced in Chapter 7, "Recursive Language Models") can inspect past sessions, cluster them by intent, and propose a `SKILL.md` for the most common cluster.

**How it works:**
- **Sessions** — a small directory of past agent transcripts.
- **RLM** — given the transcript texts, writes Python to inspect them, summarize each via a sub-LLM, cluster by intent, and pick the largest cluster.
- **Output** — a candidate `SKILL.md` written from the patterns the RLM found.

This is the same loop a human would run -- read sessions, spot the pattern, write the skill -- but the RLM does the reading and clustering for you so you only have to review the proposal.

In [ ]:
import os
from pathlib import Path

import dspy
from dotenv import load_dotenv

load_dotenv()

assert os.environ.get("OPENAI_API_KEY"), "set OPENAI_API_KEY first"

# Main LM for the RLM. The printed listing sets only the sub-LM,
# so this cell uses the book's default model.
dspy.configure(lm=dspy.LM("openai/gpt-5.6-luna"))

## Mock a session directory

In production this would be the agent's actual session log directory (a folder under `~/.claude/projects/`, or wherever your harness writes transcripts). For the notebook we'll write five short fake sessions to a local `fake_sessions/` folder next to this notebook. Four of them are the same underlying request -- "port this Postgres SQL to BigQuery" -- worded differently each time. One is an unrelated CSS question.

In [ ]:
SESSIONS = {
    "2026-04-12-billing.md": """User: this query runs in postgres but bigquery rejects it, can you fix
```sql
SELECT user_id, EXTRACT(EPOCH FROM created_at) FROM events
WHERE created_at > NOW() - INTERVAL '30 days' LIMIT 100;
```
Assistant: replaced EXTRACT(EPOCH...) with UNIX_SECONDS(created_at) and NOW()-INTERVAL with TIMESTAMP_SUB(CURRENT_TIMESTAMP(), INTERVAL 30 DAY).
""",
    "2026-04-14-funnel.md": """User: bigquery is complaining about this, came from our analytics db
```sql
SELECT date_trunc('day', ts), count(*) FROM signups GROUP BY 1;
```
Assistant: bigquery uses TIMESTAMP_TRUNC, not date_trunc. swapped it and aliased the column.
""",
    "2026-04-15-cohorts.md": """User: porting these postgres queries to BQ for the dashboard, this one breaks
```sql
SELECT user_id, ARRAY_AGG(event ORDER BY ts) FROM events GROUP BY user_id;
```
Assistant: BQ supports ARRAY_AGG but ORDER BY goes inside as ARRAY_AGG(event ORDER BY ts ASC).
""",
    "2026-04-16-retention.md": """User: another postgres-isms one, lateral join doesn't work in bigquery
```sql
SELECT u.id, e.first_seen FROM users u, LATERAL (SELECT MIN(ts) AS first_seen FROM events WHERE user_id = u.id) e;
```
Assistant: rewrote as a correlated subquery: SELECT u.id, (SELECT MIN(ts) FROM events WHERE user_id = u.id) AS first_seen FROM users u.
""",
    "2026-04-17-css.md": """User: hero section is wider than the viewport on mobile, no clue why
Assistant: the .grid container has min-width: 1200px. removed it and added overflow-x: hidden on body as a safety net.
""",
}

sessions_dir = Path("./fake_sessions"); sessions_dir.mkdir(exist_ok=True)
for name, body in SESSIONS.items():
    (sessions_dir / name).write_text(body)

print(f"wrote {len(SESSIONS)} sessions to {sessions_dir}/")

## Let the RLM read and cluster

`dspy.RLM` gives the model a Python REPL plus `llm_query()` for semantic work. The default Deno interpreter cannot read arbitrary host files, so the host application reads the selected files and passes their text as an input value. This also keeps the example portable across interpreter backends. The RLM can then summarize each intent, cluster the summaries, and return a skill for the largest cluster.

The next cell is the listing printed on page 300. `sessions_dir` comes from the cell above, and the listing reads every `*.md` file in it; change the pattern if your harness stores transcripts in another format. `max_iters=8` and `max_llm_calls=10` cap the run at eight REPL iterations and ten sub-LM calls.

In [ ]:
class DiscoverSkill(dspy.Signature):
    """Read every session transcript in `session_texts`. For each transcript,
    summarize in one sentence what the user was asking the agent to do. Group
    similar summaries. For the largest group (the dominant recurring intent),
    write a SKILL.md that would let the agent handle that request directly next
    time with YAML frontmatter (name, description) and a body of concrete rules
    drawn from how the assistant actually responded in those sessions.

    Skip groups smaller than 2 sessions. Return only the SKILL.md text.
    """
    session_texts: list[str] = dspy.InputField()
    skill_md: str = dspy.OutputField()

session_texts = [
    path.read_text() for path in sorted(sessions_dir.glob("*.md"))
]
sub_lm = dspy.LM("openai/gpt-5.6-luna")

rlm = dspy.RLM(
    DiscoverSkill,
    max_iters=8,
    max_llm_calls=10,
    sub_lm=sub_lm,
)
result = rlm(session_texts=session_texts)
print(result.skill_md)

> **Book vs. notebook:** The book prints the listing on page 300 without its output. The next cell adds a short report of the run: how many REPL steps the RLM took, how many times it called the sub-LM, and whether the code of any step failed.

With five short transcripts, the model can print everything in its first step, so it makes few sub-LM calls: often one, sometimes none. On a real session directory, where the transcripts do not fit in one printout, the sub-LM does more of the reading.

On some runs, steps fail with `Invalid Python syntax` although the model wrote valid code. DSPy 3.3.0 takes the model's code from a fenced block and stops reading at the next triple backtick. A SQL skill usually contains a fenced SQL example, so when the model types the skill text as a string in its Python code, the code is cut off at that inner fence. The model then retries, and often it rewrites the skill without fences and submits it. If it is still retrying when `max_iters=8` is reached, the listing logs `WARNING dspy.predict.rlm: RLM reached max iterations, using extract to get final output`, and the RLM writes the skill from what the earlier steps gathered. Either way, the draft is a SQL-dialect skill you can review. A draft written by this fallback can end with a stray `[[ ## completed ##]]` line, which the last cell leaves out of the saved file.

When you adapt the signature to your own sessions, one extra sentence in its instructions helps the model avoid these retries: ask it to build each code fence with ``"`" * 3`` instead of typing three backticks inside a string.

In [ ]:
# How the run went: `result.trajectory` holds one entry per REPL step
steps = result.trajectory
failed = [
    number for number, step in enumerate(steps, start=1)
    if step["output"].startswith("[Error]")
]
submitted = steps[-1]["output"].startswith("FINAL:")

print(f"{len(steps)} REPL steps (limit {rlm.max_iters}), {len(sub_lm.history)} sub-LM calls")
print("steps whose code failed:", failed or "none")
print("skill written by:", "SUBMIT()" if submitted else "the fallback after the last step")

## Review and install

With these five sessions, the `Skip groups smaller than 2 sessions` clause means the lone CSS question gets no skill, and the proposal should be a SQL-dialect skill drawn from the four Postgres-to-BigQuery sessions (page 300). The wording of the skill varies from run to run.

The RLM proposes; you decide. Read the candidate, edit anything that's off, then save it to wherever your harness picks up skills (`~/.claude/skills/<name>/SKILL.md`, `.cursor/rules/`, etc.). From here the loop from "Optimizing a SKILL.md End-to-End", the chapter's first section (`landing-page-skill-optimizer.ipynb`), takes over -- run the new skill on real requests, capture the failures, and feed those failures into `optimize_anything` to evolve it.

The last cell saves the draft to `proposed_skill.md` next to this notebook so you can review it. It does not install anything. If the draft ends with a stray `[[ ## completed ##]]` line (see the note under the listing), the cell leaves that line out of the file.

In [ ]:
import re

# A draft written by the fallback can end with a stray `[[ ## completed ##]]` line.
# Leave that line out of the saved file.
skill_text = re.sub(r"\s*\[\[ ## completed ##\s*\]\]\s*$", "", result.skill_md)

Path("./proposed_skill.md").write_text(skill_text.rstrip() + "\n")
print("saved to ./proposed_skill.md -- review before installing.")